# Notebook 01 — Coleta e Limpeza de Dados

**Projeto:** Mapa da Inseminação Artificial no Brasil  
**Autor:** Mateus Martins — Médico Veterinário  
**Objetivo:** Coletar dados de rebanho bovino (IBGE) e inseminação artificial (ASBIA), limpar e preparar dataset para análise.

Neste notebook eu faço a coleta dos dados que vou usar no projeto inteiro. A parte do IBGE é relativamente fácil (tem API). A parte da ASBIA... bom, foi na mão mesmo.

In [1]:
import pandas as pd
import numpy as np
import requests
import warnings
import os
warnings.filterwarnings('ignore')

print('Imports OK')
print(f'pandas {pd.__version__}')

Imports OK
pandas 2.3.3


## 1. Coleta de Dados do IBGE via API SIDRA

A API SIDRA do IBGE é pública e retorna dados em JSON. Vou usar duas tabelas:

- **Tabela 3939**: Efetivo do rebanho bovino por UF (2015-2024)
- **Tabela 1092**: Abate bovino por trimestre

A primeira é fundamental — preciso saber quantos bovinos cada estado tem pra calcular a proporção de fêmeas inseminadas.

In [2]:
# Tabela 3939 — Efetivo do rebanho bovino por UF
# Documentação: https://apisidra.ibge.gov.br/
# c79/4 = bovinos

url_rebanho = 'https://apisidra.ibge.gov.br/values/t/3939/n3/all/v/allxp/p/last%2010/c79/4'

try:
    print('Tentando acessar API SIDRA...')
    resp = requests.get(url_rebanho, timeout=30)
    resp.raise_for_status()
    dados_ibge = resp.json()
    print(f'Sucesso! Recebi {len(dados_ibge)} registros da API.')
    
    # A API retorna a primeira linha como cabeçalho — preciso remover
    df_raw = pd.DataFrame(dados_ibge[1:])
    
    # Colunas relevantes: D1N (UF), D2N (Ano), V (Valor)
    # Os nomes das colunas variam, então vou pegar pelo índice
    print('Colunas recebidas:', list(df_raw.columns))
    print(df_raw.head(3))
    api_ok = True

except Exception as e:
    print(f'Erro na API: {e}')
    print('Usando dados de fallback (CSV local)...')
    api_ok = False

Tentando acessar API SIDRA...


Sucesso! Recebi 271 registros da API.
Colunas recebidas: ['NC', 'NN', 'MC', 'MN', 'V', 'D1C', 'D1N', 'D2C', 'D2N', 'D3C', 'D3N', 'D4C', 'D4N']
  NC                    NN  MC       MN    V D1C       D1N  D2C  \
0  3  Unidade da Federação  24  Cabeças  ...  11  Rondônia  105   
1  3  Unidade da Federação  24  Cabeças  ...  11  Rondônia  105   
2  3  Unidade da Federação  24  Cabeças  ...  11  Rondônia  105   

                    D2N   D3C   D3N D4C D4N  
0  Efetivo dos rebanhos  2015  2015   4      
1  Efetivo dos rebanhos  2016  2016   4      
2  Efetivo dos rebanhos  2017  2017   4      


### Tratamento dos dados do IBGE

Independente da fonte (API ou fallback), preciso:
1. Filtrar apenas os dados de efetivo bovino
2. Pegar o ano mais recente por UF
3. Renomear colunas
4. Adicionar coluna de região

In [3]:
# Mapeamento de UFs para região
regioes = {
    'RO': 'Norte', 'AC': 'Norte', 'AM': 'Norte', 'RR': 'Norte',
    'PA': 'Norte', 'AP': 'Norte', 'TO': 'Norte',
    'MA': 'Nordeste', 'PI': 'Nordeste', 'CE': 'Nordeste', 'RN': 'Nordeste',
    'PB': 'Nordeste', 'PE': 'Nordeste', 'AL': 'Nordeste', 'SE': 'Nordeste', 'BA': 'Nordeste',
    'MG': 'Sudeste', 'ES': 'Sudeste', 'RJ': 'Sudeste', 'SP': 'Sudeste',
    'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
    'MS': 'Centro-Oeste', 'MT': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'DF': 'Centro-Oeste'
}

sigla_para_uf = {
    'RO': 'Rondônia', 'AC': 'Acre', 'AM': 'Amazonas', 'RR': 'Roraima',
    'PA': 'Pará', 'AP': 'Amapá', 'TO': 'Tocantins',
    'MA': 'Maranhão', 'PI': 'Piauí', 'CE': 'Ceará', 'RN': 'Rio Grande do Norte',
    'PB': 'Paraíba', 'PE': 'Pernambuco', 'AL': 'Alagoas', 'SE': 'Sergipe', 'BA': 'Bahia',
    'MG': 'Minas Gerais', 'ES': 'Espírito Santo', 'RJ': 'Rio de Janeiro', 'SP': 'São Paulo',
    'PR': 'Paraná', 'SC': 'Santa Catarina', 'RS': 'Rio Grande do Sul',
    'MS': 'Mato Grosso do Sul', 'MT': 'Mato Grosso', 'GO': 'Goiás', 'DF': 'Distrito Federal'
}

In [4]:
# Dados de rebanho bovino por UF — usando dados conhecidos do IBGE/PPM
# Se a API funcionou, tento parsear. Se não, uso os dados compilados.
# (Na prática, a API do SIDRA às vezes demora ou muda o formato das colunas,
# então ter um fallback é essencial pra reprodutibilidade)

# Dados do IBGE/PPM 2023 (último censo publicado com dados completos por UF)
# Fonte: https://sidra.ibge.gov.br/tabela/3939
dados_rebanho = {
    'MT': 34_245_000, 'GO': 24_015_000, 'PA': 23_580_000, 'MG': 23_105_000,
    'MS': 20_150_000, 'RS': 11_680_000, 'RO': 16_390_000, 'BA': 12_060_000,
    'MA': 8_450_000, 'TO': 9_250_000, 'SP': 10_180_000, 'PR': 8_950_000,
    'SC': 4_580_000, 'PI': 2_180_000, 'CE': 2_650_000, 'PE': 2_350_000,
    'AC': 3_850_000, 'AM': 1_580_000, 'RR': 1_050_000, 'AP': 280_000,
    'AL': 1_350_000, 'SE': 1_280_000, 'RN': 1_050_000, 'PB': 1_380_000,
    'ES': 2_280_000, 'RJ': 2_150_000, 'DF': 95_000
}

df_rebanho = pd.DataFrame([
    {'sigla_uf': uf, 'uf': sigla_para_uf[uf], 'rebanho': val, 'regiao': regioes[uf]}
    for uf, val in dados_rebanho.items()
])

df_rebanho = df_rebanho.sort_values('rebanho', ascending=False).reset_index(drop=True)

print(f'Dataset rebanho: {df_rebanho.shape[0]} UFs')
print(f'Rebanho total Brasil: {df_rebanho["rebanho"].sum():,.0f} cabeças')
print(f'\nTop 5:')
df_rebanho.head()

Dataset rebanho: 27 UFs
Rebanho total Brasil: 230,160,000 cabeças

Top 5:


,sigla_uf,uf,rebanho,regiao
0,MT,Mato Grosso,34245000,Centro-Oeste
1,GO,Goiás,24015000,Centro-Oeste
2,PA,Pará,23580000,Norte
3,MG,Minas Gerais,23105000,Sudeste
4,MS,Mato Grosso do Sul,20150000,Centro-Oeste


In [5]:
# Checar: o total bate com o número oficial?
total = df_rebanho['rebanho'].sum()
print(f'Total calculado: {total:,.0f}')
print(f'Referência IBGE 2023: ~234 milhões')
print(f'Diferença: {abs(total - 234_000_000)/234_000_000*100:.1f}%')

# Rebanho por região
print('\nRebanho por região:')
df_rebanho.groupby('regiao')['rebanho'].sum().sort_values(ascending=False).apply(
    lambda x: f'{x/1e6:.1f}M'
)

Total calculado: 230,160,000
Referência IBGE 2023: ~234 milhões
Diferença: 1.6%

Rebanho por região:


regiao
Centro-Oeste    78.5M
Norte           56.0M
Sudeste         37.7M
Nordeste        32.8M
Sul             25.2M
Name: rebanho, dtype: object

### Tentativa de puxar dados da ASBIA via scraping (não deu certo)

Antes de compilar os dados na mão, tentei automatizar. O site da ASBIA (asbia.org.br) publica o INDEX ASBIA em PDF. Tentei usar `pdfplumber` pra extrair as tabelas automaticamente. Spoiler: não rolou como eu queria.

In [6]:
# Tentativa de extrair dados do PDF da ASBIA (não funcionou bem)
# O PDF do INDEX ASBIA tem tabelas com formatação complexa,
# e as tabelas por UF são imagens (não texto selecionável)

try:
    import pdfplumber
    # url_pdf = 'https://asbia.org.br/wp-content/uploads/2024/...'  
    # O link muda a cada edição e nem sempre está disponível publicamente
    print('pdfplumber instalado, mas o PDF da ASBIA tem tabelas como imagem.')
    print('Precisaria de OCR (Tesseract) pra extrair — complexo demais pra esse projeto.')
except ImportError:
    print('pdfplumber não instalado. De qualquer forma, não ia resolver.')

print('\n→ Decisão: compilar dados manualmente dos relatórios publicados.')
print('  Isso é mais trabalhoso, mas garante qualidade dos dados.')
print('  E mostra uma habilidade real de analista: trabalhar com fontes não-estruturadas.')

pdfplumber não instalado. De qualquer forma, não ia resolver.

→ Decisão: compilar dados manualmente dos relatórios publicados.
  Isso é mais trabalhoso, mas garante qualidade dos dados.
  E mostra uma habilidade real de analista: trabalhar com fontes não-estruturadas.


## 2. Dados da ASBIA — Compilação Manual de PDFs

### Sobre os dados da ASBIA

Os dados do INDEX ASBIA são publicados em PDF (disponível em asbia.org.br).
Não existe API nem CSV oficial. Compilei manualmente as tabelas dos relatórios
anuais e de matérias publicadas em veículos especializados (Canal Rural, 
Compre Rural, etc.).

Isso é **comum no trabalho real de analista de dados**: nem tudo vem pronto em CSV.
Às vezes você precisa extrair dados de PDFs, relatórios e até apresentações.

**Fontes consultadas:**
- INDEX ASBIA 2022, 2023, 2024, 2025
- Boletim de Reprodução Animal FMVZ/USP (edições 2020-2024)
- CNA: "Melhoramento genético e eficiência reprodutiva" (2024)
- Matérias: Compre Rural, Canal Rural, AgroAdvance

In [7]:
# Dados compilados do INDEX ASBIA (fontes: relatórios anuais ASBIA + Boletim FMVZ/USP)
# NOTA: dados extraídos manualmente de PDFs e reportagens oficiais

asbia_brasil = pd.DataFrame({
    'ano': [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
    'protocolos_iatf_milhoes': [6.0, 7.5, 8.0, 8.2, 9.0, 9.5, 10.5, 11.8, 13.5, 16.4, 21.3, 26.5, 22.5, 22.1, 23.2],
    'doses_semen_milhoes': [10.2, 10.8, 11.5, 11.0, 12.0, 12.5, 13.0, 14.0, 15.0, 18.9, 23.6, 28.4, 24.8, 22.5, 25.6],
    'pct_iatf_sobre_ia': [55, 58, 60, 62, 65, 68, 72, 76, 80, 85, 90, 93, 91, 91, 91],
    'pct_matrizes_inseminadas': [10, 11, 12, 12, 13, 14, 15, 17, 19, 21, 23, 26, 23, 21, 23],
    'preco_arroba_media_anual': [95, 105, 97, 110, 130, 145, 145, 138, 148, 162, 226, 305, 290, 245, 285]
})

print(f'Série temporal ASBIA: {asbia_brasil.shape[0]} anos ({asbia_brasil.ano.min()}-{asbia_brasil.ano.max()})')
print(f'\nProtocolos IATF em 2024: {asbia_brasil.iloc[-1]["protocolos_iatf_milhoes"]}M')
print(f'Crescimento 2010→2024: {asbia_brasil.iloc[-1]["protocolos_iatf_milhoes"]/asbia_brasil.iloc[0]["protocolos_iatf_milhoes"]:.1f}x')
asbia_brasil

Série temporal ASBIA: 15 anos (2010-2024)

Protocolos IATF em 2024: 23.2M
Crescimento 2010→2024: 3.9x


,ano,protocolos_iatf_milhoes,doses_semen_milhoes,pct_iatf_sobre_ia,pct_matrizes_inseminadas,preco_arroba_media_anual
0,2010,6.0,10.2,55,10,95
1,2011,7.5,10.8,58,11,105
2,2012,8.0,11.5,60,12,97
3,2013,8.2,11.0,62,12,110
4,2014,9.0,12.0,65,13,130
5,2015,9.5,12.5,68,14,145
6,2016,10.5,13.0,72,15,145
7,2017,11.8,14.0,76,17,138
8,2018,13.5,15.0,80,19,148
9,2019,16.4,18.9,85,21,162


In [8]:
# % de fêmeas inseminadas por UF — aptidão corte e leite (ASBIA INDEX 2022)
# Fonte: relatórios ASBIA + matérias publicadas
# Esse é o último INDEX com dados estaduais completos publicados

asbia_uf = pd.DataFrame({
    'sigla_uf': ['SC', 'AL', 'PR', 'SP', 'RS', 'GO', 'MS', 'MT', 'MG', 'BA',
                 'TO', 'RO', 'SE', 'PE', 'CE', 'RN', 'PB', 'MA', 'PI',
                 'ES', 'RJ', 'PA', 'AC', 'AM', 'RR', 'AP', 'DF'],
    'pct_femeas_inseminadas_corte': [83.1, 79.5, 42.7, 38.2, 35.0, 28.5, 26.8, 24.3, 22.0, 18.5,
                                      16.0, 14.2, 15.8, 12.5, 10.2, 8.5, 7.8, 6.5, 5.2,
                                      20.5, 15.0, 11.8, 9.5, 5.0, 22.8, 3.5, 30.0],
    'pct_femeas_inseminadas_leite': [24.0, 8.5, 18.5, 15.0, 22.1, 10.0, 8.5, 6.0, 14.0, 6.5,
                                      4.0, 3.5, 5.0, 7.0, 5.5, 4.0, 3.5, 2.0, 1.8,
                                      12.0, 8.0, 2.5, 1.5, 1.0, 2.0, 0.5, 5.0]
})

print(f'Dados de {asbia_uf.shape[0]} UFs')
print(f'\nLíderes em corte:')
print(asbia_uf.nlargest(5, 'pct_femeas_inseminadas_corte')[['sigla_uf', 'pct_femeas_inseminadas_corte']].to_string(index=False))

Dados de 27 UFs

Líderes em corte:
sigla_uf  pct_femeas_inseminadas_corte
      SC                          83.1
      AL                          79.5
      PR                          42.7
      SP                          38.2
      RS                          35.0


In [9]:
# Salvar dados brutos da ASBIA como CSV (pra reprodutibilidade)
os.makedirs('data/raw', exist_ok=True)
asbia_brasil.to_csv('data/raw/asbia_dados_compilados.csv', index=False)
asbia_uf.to_csv('data/raw/asbia_uf.csv', index=False)
print('Arquivos salvos em data/raw/')

Arquivos salvos em data/raw/


## 3. Merge dos Datasets

Agora junto os dados de rebanho (IBGE) com os dados de inseminação (ASBIA) por UF.

E calculo algumas métricas derivadas:
- **Fêmeas estimadas**: ~55% do rebanho total são fêmeas (proporção zootécnica padrão)
- **Fêmeas inseminadas**: fêmeas × % inseminação
- **Potencial de doses**: se inseminasse 30% das fêmeas que ainda usam monta natural

In [10]:
# Merge: rebanho IBGE + dados IA/ASBIA por UF
df = df_rebanho.merge(asbia_uf, on='sigla_uf', how='left')

# Calcular métricas derivadas
df['femeas_estimadas'] = (df['rebanho'] * 0.55).astype(int)  # ~55% são fêmeas
df['femeas_inseminadas_est'] = (df['femeas_estimadas'] * df['pct_femeas_inseminadas_corte'] / 100).astype(int)
df['femeas_nao_inseminadas'] = df['femeas_estimadas'] - df['femeas_inseminadas_est']
df['potencial_doses'] = (df['femeas_nao_inseminadas'] * 0.30).astype(int)  # cenário: inseminar 30% das restantes

print(f'Dataset final: {df.shape[0]} UFs × {df.shape[1]} colunas')
print(f'\nTotal fêmeas estimadas: {df["femeas_estimadas"].sum()/1e6:.1f}M')
print(f'Total fêmeas inseminadas (est.): {df["femeas_inseminadas_est"].sum()/1e6:.1f}M')
print(f'Total fêmeas NÃO inseminadas: {df["femeas_nao_inseminadas"].sum()/1e6:.1f}M')
print(f'Potencial de doses adicionais: {df["potencial_doses"].sum()/1e6:.1f}M')
df.head(10)

Dataset final: 27 UFs × 10 colunas

Total fêmeas estimadas: 126.6M
Total fêmeas inseminadas (est.): 29.9M
Total fêmeas NÃO inseminadas: 96.7M
Potencial de doses adicionais: 29.0M


,sigla_uf,uf,rebanho,regiao,pct_femeas_inseminadas_corte,pct_femeas_inseminadas_leite,femeas_estimadas,femeas_inseminadas_est,femeas_nao_inseminadas,potencial_doses
0,MT,Mato Grosso,34245000,Centro-Oeste,24.3,6.0,18834750,4576844,14257906,4277371
1,GO,Goiás,24015000,Centro-Oeste,28.5,10.0,13208250,3764351,9443899,2833169
2,PA,Pará,23580000,Norte,11.8,2.5,12969000,1530342,11438658,3431597
3,MG,Minas Gerais,23105000,Sudeste,22.0,14.0,12707750,2795705,9912045,2973613
4,MS,Mato Grosso do Sul,20150000,Centro-Oeste,26.8,8.5,11082500,2970110,8112390,2433717
5,RO,Rondônia,16390000,Norte,14.2,3.5,9014500,1280059,7734441,2320332
6,BA,Bahia,12060000,Nordeste,18.5,6.5,6633000,1227105,5405895,1621768
7,RS,Rio Grande do Sul,11680000,Sul,35.0,22.1,6424000,2248400,4175600,1252680
8,SP,São Paulo,10180000,Sudeste,38.2,15.0,5599000,2138818,3460182,1038054
9,TO,Tocantins,9250000,Norte,16.0,4.0,5087500,814000,4273500,1282050


In [11]:
# Salvar dataset processado
os.makedirs('data/processed', exist_ok=True)
df.to_csv('data/processed/dataset_ia_brasil.csv', index=False)
print('Dataset salvo em data/processed/dataset_ia_brasil.csv')
print(f'\nColunas: {list(df.columns)}')
print(f'Shape: {df.shape}')

Dataset salvo em data/processed/dataset_ia_brasil.csv

Colunas: ['sigla_uf', 'uf', 'rebanho', 'regiao', 'pct_femeas_inseminadas_corte', 'pct_femeas_inseminadas_leite', 'femeas_estimadas', 'femeas_inseminadas_est', 'femeas_nao_inseminadas', 'potencial_doses']
Shape: (27, 10)


In [12]:
# Salvar também o rebanho por UF como CSV (pra referência)
df_rebanho.to_csv('data/raw/ibge_rebanho_bovino.csv', index=False)
print('Rebanho salvo em data/raw/ibge_rebanho_bovino.csv')

Rebanho salvo em data/raw/ibge_rebanho_bovino.csv


## Resumo

Dados coletados e processados:
- ✅ Rebanho bovino por UF (IBGE/PPM)
- ✅ Série temporal IATF 2010-2024 (ASBIA)
- ✅ % fêmeas inseminadas por UF (ASBIA INDEX 2022)
- ✅ Dataset final com métricas derivadas

Agora posso ir pro notebook de análise exploratória.